In [11]:
import pandas as pd
import re

# --- Step 1: Read input CSVs ---
file1 = pd.read_csv("./pred/test_predictions_bn_greedy.csv")
file2 = pd.read_csv("./pred/english_to_bengali_seq2seq_style_fixed1.csv")

# Ensure consistent column names
file1.columns = ["ID", "Translation"]
file2.columns = ["ID", "Translation"]

# Normalize IDs to string
file1["ID"] = file1["ID"].astype(str).str.strip()
file2["ID"] = file2["ID"].astype(str).str.strip()

# --- Step 2: Clean file-2 translations (remove English letters and numbers) ---
file2["Translation"] = file2["Translation"].apply(
    lambda x: re.sub(r"[A-Za-z0-9]", "", str(x)).strip()
)

# --- Step 3: Merge data preserving file-1 order ---
file2_dict = dict(zip(file2["ID"], file2["Translation"]))

merged_data = []
for _, row in file1.iterrows():
    id_ = row["ID"]
    print(f"Processing ID: {id_} and is {id_ in file2_dict} present")
    if id_ in file2_dict:
        merged_data.append((id_, file2_dict[id_]))
    else:
        merged_data.append((id_, row["Translation"]))

# --- Step 4: Create merged DataFrame ---
new_df = pd.DataFrame(merged_data, columns=["ID", "Translation"])

# --- Step 5: Save to answer.csv (temporary CSV first) ---
new_df.to_csv("answer.csv", index=False, encoding="utf-8")

# --- Step 6: Rewrite in required tab-separated format ---
filtered_data = pd.read_csv("answer.csv")
with open("answer.csv", "w", encoding="utf-8") as f:
    f.writelines("ID\tTranslation\n")
    for i in range(filtered_data.shape[0]):
        f.writelines(f'{filtered_data["ID"][i]}\t"{filtered_data["Translation"][i]}"\n')

# --- Step 7: Verify lengths match ---
file1_check = pd.read_csv("./pred/test_predictions_bn_greedy.csv")
answer_check = pd.read_csv("answer.csv", sep="\t")

print("Length of file-1:", len(file1_check))
print("Length of answer.csv:", len(answer_check))

# Assertion to ensure equality
assert len(file1_check) == len(answer_check), "❌ Length mismatch between file-1 and answer.csv!"
print("✅ Assertion passed: Lengths match perfectly.")


Processing ID: 177039 and is True present
Processing ID: 177040 and is True present
Processing ID: 177041 and is True present
Processing ID: 177042 and is True present
Processing ID: 177043 and is True present
Processing ID: 177044 and is True present
Processing ID: 177045 and is True present
Processing ID: 177046 and is True present
Processing ID: 177047 and is True present
Processing ID: 177048 and is True present
Processing ID: 177049 and is True present
Processing ID: 177050 and is True present
Processing ID: 177051 and is True present
Processing ID: 177052 and is True present
Processing ID: 177053 and is True present
Processing ID: 177054 and is True present
Processing ID: 177055 and is True present
Processing ID: 177056 and is True present
Processing ID: 177057 and is True present
Processing ID: 177058 and is True present
Processing ID: 177059 and is True present
Processing ID: 177060 and is True present
Processing ID: 177061 and is True present
Processing ID: 177062 and is True 

In [5]:
import re

input_path = "./pred/english_to_bengali_seq2seq_style_fixed.csv"
output_path = "./pred/english_to_bengali_seq2seq_style_fixed1.csv"  # overwrite same file

cleaned_lines = []
buffer = ""

with open(input_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()

        # If this line looks incomplete (odd number of quotes), it’s likely broken
        if line.count('"') % 2 != 0:
            buffer += line + " "  # accumulate multi-line text
            continue

        # If there was a buffer waiting and this line completes it
        if buffer:
            buffer += line
            line = buffer
            buffer = ""  # reset buffer

        # Extract only the first two comma-separated quoted fields
        match = re.match(r'^"?(.*?)"?,\s*"?([^"]*)"?$', line)
        if match:
            id_val, trans_val = match.groups()
            # remove stray newlines or quotes inside
            id_val = id_val.strip().replace('"', '')
            trans_val = trans_val.strip().replace('"', '')
            cleaned_lines.append(f'"{id_val}","{trans_val}"\n')
        else:
            # Skip or log lines that don't fit
            continue

# Write cleaned output back
with open(output_path, "w", encoding="utf-8") as f:
    f.writelines(cleaned_lines)

print(f"✅ Cleaned and fixed malformed CSV saved to: {output_path}")
print(f"Total valid lines written: {len(cleaned_lines)}")


✅ Cleaned and fixed malformed CSV saved to: ./pred/english_to_bengali_seq2seq_style_fixed1.csv
Total valid lines written: 11569


In [ ]:
import pandas as pd

# Step 1: Combine Bengali & Hindi predictions
df1 = pd.read_csv("answer_ai_bengali.csv",sep="\t")  # Bengali
df2 = pd.read_csv("./pred/test_predictions_hi_beam_search.csv")  # Hindi
df3 = pd.concat([df1, df2])

# Step 2: Save combined data to 'answer.csv'
df3.to_csv('answer.csv', index=False)

# Step 3: Reformat file with tab-separated output
filtered_data = pd.read_csv("answer.csv")
with open("answer.csv", "w", encoding="utf-8") as f:
    f.writelines("ID\tTranslation\n")
    for i in range(filtered_data.shape[0]):
        f.writelines(f'{filtered_data["ID"][i]}\t"{filtered_data["Translation"][i]}"\n')

# Step 4: Compare with answerB.csv
answer_a = pd.read_csv("answer.csv", sep="\t")  # your file
# Step 6: Find IDs with missing or NaN translations in your file
na_predictions = answer_a[answer_a["Translation"].isna() | (answer_a["Translation"].astype(str).str.strip() == "")]

# Step 7: Save both results
print(f"NA / empty translations found: {len(na_predictions)} (saved to na_predictions.csv)")

NA / empty translations found: 13 (saved to na_predictions.csv)


#### Hindi

In [16]:
import re

input_path = "./ai/hindi.csv"
output_path = "./ai/hindi_clean.csv"

clean_lines = []
with open(input_path, "r", encoding="utf-8") as f:
    buffer = ""
    for line in f:
        line = line.strip()
        # Accumulate until a full quote pair is seen
        buffer += line + " "
        if buffer.count('"') % 2 == 0:  # even number of quotes = complete line
            clean_lines.append(buffer.strip())
            buffer = ""

# Repair missing closing quote (if still open)
fixed_lines = []
for l in clean_lines:
    if not l.endswith('"'):
        l = l + '"'
    fixed_lines.append(l)

# Keep only well-formed CSV-like rows
final_lines = []
for l in fixed_lines:
    if l.startswith('"') and '","' in l:
        parts = l.split('","', 1)
        if len(parts) == 2:
            id_val = parts[0].strip('"')
            text_val = parts[1].rstrip('"').replace('\n', ' ').strip()
            final_lines.append(f'"{id_val}","{text_val}"')

with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n".join(final_lines))

print(f"✅ Cleaned file saved to {output_path}")
print(f"🧹 Total cleaned lines: {len(final_lines)}")

✅ Cleaned file saved to ./ai/hindi_clean.csv
🧹 Total cleaned lines: 14726


In [28]:
import pandas as pd
import re

# --- Step 1: Read input CSVs ---
file1 = pd.read_csv("./ai/test_predictions_hi_beam_search.csv")
file2 = pd.read_csv("./ai/hindi_clean.csv")

# Ensure consistent column names
file1.columns = ["ID", "Translation"]
file2.columns = ["ID", "Translation"]

# --- Step 2: Normalize IDs ---
file1["ID"] = file1["ID"].astype(str).str.strip()
file2["ID"] = file2["ID"].astype(str).str.strip()

# --- Step 3: Clean Hindi translations ---
def clean_hindi_text_strict(text):
    """
    Cleans Hindi text strictly:
    ✅ Keeps only:
        - Hindi Devanagari letters (ऀ–ॿ)
        - Hindi numerals (०–९)
        - Whitespace and a few punctuation marks: । , ! ?
    🚫 Removes everything else (English, emojis, Latin chars, weird symbols)
    """
    text = str(text)

    # Substitute any non-Hindi character with empty string
    text = re.sub(r"[^\u0900-\u097F\u0966-\u096F\s\u0964,!?]", "", text)

    # Normalize spacing (remove extra spaces)
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Apply strict cleaning
file2["Translation"] = file2["Translation"].apply(clean_hindi_text_strict)

# --- Step 4: Merge data preserving file-1 order ---
file2_dict = dict(zip(file2["ID"], file2["Translation"]))

merged_data = []
for _, row in file1.iterrows():
    id_ = row["ID"]
    print(f"Processing ID: {id_} and is {id_ in file2_dict} present")
    if id_ in file2_dict:
        merged_data.append((id_, file2_dict[id_]))
    else:
        merged_data.append((id_, row["Translation"]))

# --- Step 5: Create merged DataFrame ---
new_df = pd.DataFrame(merged_data, columns=["ID", "Translation"])

# --- Step 6: Save to intermediate CSV ---
new_df.to_csv("./ai/hindi_answer_temp.csv", index=False, encoding="utf-8")

# --- Step 7: Rewrite in tab-separated format ---
filtered_data = pd.read_csv("./ai/hindi_answer_temp.csv")
with open("./ai/hindi_answer.csv", "w", encoding="utf-8") as f:
    f.writelines("ID\tTranslation\n")
    for i in range(filtered_data.shape[0]):
        f.writelines(f'{filtered_data["ID"][i]}\t"{filtered_data["Translation"][i]}"\n')

# --- Step 8: Verify lengths match ---
file1_check = pd.read_csv("./ai/test_predictions_hi_beam_search.csv")
answer_check = pd.read_csv("./ai/hindi_answer.csv", sep="\t")

print("Length of file-1:", len(file1_check))
print("Length of hindi_answer.csv:", len(answer_check))

assert len(file1_check) == len(answer_check), "❌ Length mismatch between file-1 and hindi_answer.csv!"
print("✅ Assertion passed: Lengths match perfectly.")


Processing ID: 540139 and is True present
Processing ID: 540140 and is True present
Processing ID: 540141 and is True present
Processing ID: 540142 and is True present
Processing ID: 540143 and is True present
Processing ID: 540144 and is True present
Processing ID: 540145 and is True present
Processing ID: 540146 and is True present
Processing ID: 540147 and is True present
Processing ID: 540148 and is True present
Processing ID: 540149 and is True present
Processing ID: 540150 and is True present
Processing ID: 540151 and is True present
Processing ID: 540152 and is True present
Processing ID: 540153 and is True present
Processing ID: 540154 and is True present
Processing ID: 540155 and is True present
Processing ID: 540156 and is True present
Processing ID: 540157 and is True present
Processing ID: 540158 and is True present
Processing ID: 540159 and is True present
Processing ID: 540160 and is True present
Processing ID: 540161 and is True present
Processing ID: 540162 and is True 

AssertionError: ❌ Length mismatch between file-1 and hindi_answer.csv!

In [ ]:
file1 = pd.read_csv("./ai/test_predictions_hi_beam_search.csv")
file2 = pd.read_csv("./ai/hindi_answer.csv",sep="\t")

# Ensure consistent column names
file1.columns = ["ID", "Translation"]
file2.columns = ["ID", "Translation"]

# --- Step 2: Normalize IDs ---
file1["ID"] = file1["ID"].astype(str).str.strip()
file2["ID"] = file2["ID"].astype(str).str.strip()

print(set(file1["ID"]) - set(file2["ID"]))


{'562303'}


KeyboardInterrupt: 

In [ ]:
print(set(file1["ID"]) - set(file2["ID"]))

In [ ]:
import pandas as pd

# Step 1: Combine Bengali & Hindi predictions
df1 = pd.read_csv("./ai/answer_ai_bengali.csv",sep="\t")  # Bengali
df2 = pd.read_csv("./ai/hindi_answer.csv",sep="\t")  # Hindi
df3 = pd.concat([df1, df2])

# Step 2: Save combined data to 'answer.csv'
df3.to_csv('answer.csv', index=False)

# Step 3: Reformat file with tab-separated output
filtered_data = pd.read_csv("answer.csv")
with open("answer.csv", "w", encoding="utf-8") as f:
    f.writelines("ID\tTranslation\n")
    for i in range(filtered_data.shape[0]):
        f.writelines(f'{filtered_data["ID"][i]}\t"{filtered_data["Translation"][i]}"\n')

# Step 4: Compare with answerB.csv
answer_a = pd.read_csv("answer.csv", sep="\t")  # your file
# Step 6: Find IDs with missing or NaN translations in your file
na_predictions = answer_a[answer_a["Translation"].isna() | (answer_a["Translation"].astype(str).str.strip() == "")]

# Step 7: Save both results
print(f"NA / empty translations found: {len(na_predictions)} (saved to na_predictions.csv)")

FileNotFoundError: [Errno 2] No such file or directory: '.ai/answer_ai_bengali.csv'